# 🚢 Titanic · Who Survives? — starting kit

**Python & ML Workshop · American University in Dubai**

Your mission: predict which of the **179 test passengers** survived, then
upload your predictions to the Codabench leaderboard.

The plan is the one from the workshop: **load → explore → prepare → split →
fit → predict → score → submit.** Run every cell once to make a valid
submission in under a minute, then make it better.

## 1 · Load the data

In [ ]:
import os, zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Works in the starting kit folder, and in Google Colab (reads from GitHub).
SRC = "." if os.path.exists("train.csv") else "https://raw.githubusercontent.com/YouvenZ/ml-workshop-2h/main/codabench/bundle/input_data"
train = pd.read_csv(f"{SRC}/train.csv")
test  = pd.read_csv(f"{SRC}/test.csv")
print(train.shape, test.shape)
train.head()

## 2 · Explore: the first three questions

In [ ]:
print(train['Survived'].value_counts(normalize=True).round(3))   # the baseline
train.isna().sum()

In [ ]:
train.groupby('Sex')['Survived'].mean().plot(kind='bar', color=['#E07A2C', '#2E5CA8'])
plt.ylabel('survival rate'); plt.title('Survival rate by sex'); plt.show()

## 3 · Two baselines to beat

| baseline | idea |
|---|---|
| *Nobody survived* | predict 0 for everyone |
| *Only women survived* | predict 1 if `Sex == 'female'` |

A model is only interesting if it beats these.

## 4 · Prepare the features (numbers only, no gaps)

In [ ]:
def prepare(df, age_median):
    X = pd.DataFrame()
    X['Pclass']    = df['Pclass']
    X['is_female'] = (df['Sex'] == 'female').astype(int)
    X['Age']       = df['Age'].fillna(age_median)   # fill with the TRAIN median
    X['Fare']      = df['Fare'].fillna(0)
    X['family']    = df['SibSp'] + df['Parch']
    return X

age_median = train['Age'].median()
X, y = prepare(train, age_median), train['Survived']
X_test = prepare(test, age_median)
X.head()

## 5 · Split → fit → predict → score (on your own validation set)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

X_tr, X_val, y_tr, y_val = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)
model = LogisticRegression(max_iter=1000).fit(X_tr, y_tr)
print('validation accuracy:', round(accuracy_score(y_val, model.predict(X_val)), 3))

## 6 · Make it better

Ideas, from easy to clever:

- try `KNeighborsClassifier` (scale the features first!) or `RandomForestClassifier`
- add `Embarked` (one-hot with `pd.get_dummies`)
- extract the **title** from `Name` (Mr, Mrs, Miss, Master…): it encodes sex *and* age
- does having a `Cabin` number tell you anything?
- use `cross_val_score(model, X, y, cv=5)` to compare ideas honestly

## 7 · Train on all of train.csv, predict test.csv, write the submission

In [ ]:
model = LogisticRegression(max_iter=1000).fit(X, y)

sub = pd.DataFrame({'PassengerId': test['PassengerId'],
                    'Survived': model.predict(X_test)})
sub.to_csv('predictions.csv', index=False)
with zipfile.ZipFile('submission.zip', 'w') as z:
    z.write('predictions.csv')
print(sub.shape, sub['Survived'].mean().round(3), '→ submission.zip is ready')

## 8 · Submit

Codabench → **My Submissions** → upload **`submission.zip`**. Your score and a
detailed report (confusion matrix, baselines, accuracy by group) appear within
a minute. In Colab, download the zip first from the 📁 Files panel.